In [1]:
%pip install langsmith
%pip install chromadb
%pip install sentence-transformers
%pip install langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install google-genai


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from google import genai
from google.genai import types

from langsmith import Client, wrappers

import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
GOOGLE_API_KEY = "YOUR_APY_KEY"

gemini_client = wrappers.wrap_gemini(
    genai.Client(api_key=GOOGLE_API_KEY)
)

ls_client = Client()

In [6]:
documents = [

"""
Paris is the capital city of France.
It has a population of about 2 million.
The Eiffel Tower is located there.
""",

"""
Tokyo is the capital of Japan.
It is one of the largest cities in the world.
""",

"""
New Delhi is the capital of India.
It is located in northern India.
"""
]

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20
)

chunks = []

for doc in documents:
    chunks.extend(splitter.split_text(doc))

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

client = chromadb.Client()

collection = client.create_collection(
    "rag-demo"
)

embeddings = embedding_model.encode(chunks).tolist()

collection.add(

    ids=[str(i) for i in range(len(chunks))],

    documents=chunks,

    embeddings=embeddings

)

def retrieve(question, k=2):

    q_embedding = embedding_model.encode(question).tolist()

    results = collection.query(

        query_embeddings=[q_embedding],

        n_results=k

    )

    return results["documents"][0]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2305.10it/s]


In [8]:
SYSTEM_PROMPT = """
Answer ONLY from the provided context.

If the answer isn't present,
say you don't know.
"""

In [11]:
def rag(question):

    docs = retrieve(question)

    context = "\n\n".join(docs)

    response = gemini_client.models.generate_content(

        model="gemini-2.5-flash",

        contents=f"""

Context:

{context}

Question:

{question}

""",

        config=types.GenerateContentConfig(

            system_instruction=SYSTEM_PROMPT,

            temperature=0

        )

    )

    return {

        "response": response.text,

        "context": context

    }

In [12]:
def ls_target(inputs):

    result = rag(inputs["question"])

    return {

        "response": result["response"],

        "context": result["context"]

    }

In [14]:
judge_system = """
You are an expert evaluator.

Return ONLY:

CORRECT

or

INCORRECT
"""

def correctness(inputs, outputs, reference_outputs):

    prompt=f"""

Question:

{inputs['question']}

Ground Truth:

{reference_outputs['answer']}

Prediction:

{outputs['response']}

"""

    response = gemini_client.models.generate_content(

        model="gemini-2.5-flash",

        contents=prompt,

        config=types.GenerateContentConfig(

            system_instruction=judge_system,

            temperature=0

        )

    )

    return response.text.strip().upper()=="CORRECT"



In [15]:
faithfulness_prompt="""
Determine whether the answer is fully supported
by the provided context.

Return ONLY

YES

or

NO
"""
def faithfulness(inputs, outputs):

    prompt=f"""

Context:

{outputs['context']}

Answer:

{outputs['response']}

"""

    response = gemini_client.models.generate_content(

        model="gemini-2.5-flash",

        contents=prompt,

        config=types.GenerateContentConfig(

            system_instruction=faithfulness_prompt,

            temperature=0

        )

    )

    return response.text.strip().upper()=="YES"


In [16]:
relevance_prompt="""
Is the answer relevant to the user's question?

Return ONLY

YES

or

NO
"""

def answer_relevance(inputs, outputs):

    prompt=f"""

Question:

{inputs['question']}

Answer:

{outputs['response']}

"""

    response = gemini_client.models.generate_content(

        model="gemini-2.5-flash",

        contents=prompt,

        config=types.GenerateContentConfig(

            system_instruction=relevance_prompt,

            temperature=0

        )

    )

    return response.text.strip().upper()=="YES"


dataset_name = "RAG Evaluation"

# Create in LangSmith or use existing dataset

"""
Question -> Answer

What is the capital of France?

Paris

What is the capital of India?

New Delhi

...
"""



'\nQuestion -> Answer\n\nWhat is the capital of France?\n\nParis\n\nWhat is the capital of India?\n\nNew Delhi\n\n...\n'

In [18]:
experiment = ls_client.evaluate(

    ls_target,

    data=dataset_name,

    evaluators=[

        correctness,

        faithfulness,

        answer_relevance

    ],

    experiment_prefix="Gemini-RAG",

    max_concurrency=1

)

LangSmithAuthError: Authentication failed for /datasets. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/datasets?limit=1&name=RAG+Evaluation', '{"detail":"Invalid token"}')